# Visual-language temporal boundary alignment
This notebook is self-bootstrapping. Because the source repository is private, add a Colab secret named `GITHUB_TOKEN` with read access to `sanay-tralshawala/video-language-temporal-boundaries` (key icon in the left sidebar), or enter a token when prompted. The token is used only for cloning and is not saved in Git configuration.

In [1]:
# Clone the private source repository, enter it, and install dependencies.
from pathlib import Path
import base64
import getpass
import os
import subprocess
import sys

REPO = 'sanay-tralshawala/video-language-temporal-boundaries'
REPO_DIR = Path('/content/video-language-temporal-boundaries')

if not (REPO_DIR / 'requirements-colab.txt').exists():
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not token:
        token = getpass.getpass('GitHub token with read access to the private repository: ')
    if not token:
        raise RuntimeError('A GitHub token is required because this repository is private.')
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    command = ['git', '-c', f'http.extraHeader=AUTHORIZATION: basic {credential}', 'clone', '--depth', '1', f'https://github.com/{REPO}.git', str(REPO_DIR)]
    completed = subprocess.run(command)
    token = credential = None
    if completed.returncode != 0:
        raise RuntimeError('Clone failed. Check that GITHUB_TOKEN can read the private repository.')

os.chdir(REPO_DIR)
requirements = REPO_DIR / 'requirements-colab.txt'
pipeline_file = REPO_DIR / 'video_boundary_pipeline.py'
if not requirements.is_file() or not pipeline_file.is_file():
    raise FileNotFoundError(f'Incomplete checkout at {REPO_DIR}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('Repository ready:', REPO_DIR)
print('Requirements file:', requirements)

Repository ready: /content/video-language-temporal-boundaries
Requirements file: /content/video-language-temporal-boundaries/requirements-colab.txt


In [2]:
# Mount persistent storage and report the accelerator.
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4


## Two-video trial dataset
Run the next cell to download one official Video-MME archive to Colab, extract only two long videos and their SRT subtitles, and delete the archive. This needs about 7 GiB free while downloading and leaves roughly 550 MiB in `/content`. Rerunning the cell reuses completed pairs.

In [3]:
# Download two official long-video pairs directly to this Colab runtime.
from download_trial_subset import download_two_video_trial

DATASET_DIR = str(download_two_video_trial('/content/Video-MME-two-video-trial'))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


videos_chunked_15.zip: reconstructing file:   0%|          |  0.00B / 5.28GB            

videos_chunked_15.zip: downloading bytes:           |  0.00B            

subtitle.zip: reconstructing file:   0%|          |  0.00B / 7.50MB            

subtitle.zip: downloading bytes:           |  0.00B            

Extracting TGom0uiW130.mp4
Extracting sDWOsWawxPc.mp4
Removed downloaded ZIP archives after successful extraction.
Trial dataset ready: /content/Video-MME-two-video-trial


## Dataset configuration
For the trial, keep `DATASET_DIR` from the preceding download cell. To use your own data instead, set it to a Drive directory containing same-stem pairs such as `B6fvT2LKEDI.mp4` and `B6fvT2LKEDI.srt`.

In [4]:
# Edit this configuration block.
from pathlib import Path
from video_boundary_pipeline import discover_videos

DATASET_DIR = globals().get('DATASET_DIR', '/content/drive/MyDrive/Video-MME-long-subset')
OUTPUT_DIR = '/content/drive/MyDrive/video_boundary_results'
VIDEO_LIST = None       # or ['B6fvT2LKEDI.mp4', ...]
METADATA_CSV = None     # optional: video_path, subtitle_path[, video_id]
FRAME_INTERVAL = 2.0
SUBTITLE_WINDOW = 15.0

if not Path(DATASET_DIR).is_dir():
    raise FileNotFoundError(f'DATASET_DIR does not exist: {DATASET_DIR}')
pairs = discover_videos(DATASET_DIR, VIDEO_LIST, METADATA_CSV, max_videos=30)
if not pairs:
    raise RuntimeError('No same-stem video/subtitle pairs found. See the dataset configuration cell above.')
print(f'Validated {len(pairs)} video/subtitle pairs')
for pair in pairs[:5]:
    print(' ', pair.video_path.name, '+', pair.subtitle_path.name)

Validated 2 video/subtitle pairs
  TGom0uiW130.mp4 + TGom0uiW130.srt
  sDWOsWawxPc.mp4 + sDWOsWawxPc.srt


In [5]:
# Run the complete cached analysis.
from video_boundary_pipeline import PipelineConfig, run_pipeline

config = PipelineConfig(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_DIR,
    frame_interval=FRAME_INTERVAL, subtitle_window=SUBTITLE_WINDOW,
    max_videos=30,
)
results = run_pipeline(config, video_list=VIDEO_LIST, metadata_csv=METADATA_CSV)
display(results['alignment'])
display(results['stats'])

Found 2 video/subtitle pairs
Device: cuda
GPU: Tesla T4 (14.6 GiB)


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[1/2] TGom0uiW130
[2/2] sDWOsWawxPc


,tolerance_seconds,visual_alignment_fraction,text_alignment_fraction
0,5.0,0.090909,0.205882
1,10.0,0.194805,0.441176
2,20.0,0.311688,0.676471


,video_id,duration_seconds,subtitle_coverage,average_visual_change,visual_peaks,text_peaks,group
0,TGom0uiW130,3169.04,0.792634,0.419429,43,21,low speech / high visual change
1,sDWOsWawxPc,1933.36,0.930041,0.299151,34,13,high speech / low visual change
